# Wizualizacja danych genomicznych

## Wprowadzenie

W tym notatniku przeprowadzimy analizę i wizualizację cech sekwencji DNA, takich jak zawartość GC i asymetria GC, wykorzystując technikę przesuwnych okien. Następnie dane zwizualizujemy przy pomocy [Circos](http://circos.ca/) - popularnego narzędzia umożliwiającego kolistą reprezentację danych genomicznych. Pakiet Circos jest niezwykle elastyczny i doskonale udokumentowany, [z wieloma przykładami i dokładnymi opisami funkcjonalności.](http://circos.ca/documentation/) 
![http://circos.ca/intro/genomic_data/](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41588-021-00922-y/MediaObjects/41588_2021_922_Fig1_HTML.png)
*https://doi.org/10.1038/s41588-021-00922-y*

## Import Bibliotek

Przed rozpoczęciem analizy danych i wizualizacji importujemy kilka kluczowych bibliotek:
1. sys: Umożliwia interakcję z interpreterem Pythona i zarządzanie argumentami wiersza poleceń.
2. pandas (pd): Służy do łatwego wczytywania i przetwarzania danych w formie tabelarycznej.
3. numpy (np): Zawiera narzędzia do obliczeń numerycznych i operacji na danych wielowymiarowych.
4. plotnine: Umożliwia tworzenie wykresów i wizualizację danych, oparta jest na gramatyce pakietu ggplot2 z języka R.

In [1]:
import sys
import pandas as pd
import numpy as np
from plotnine import *

## Zdefiniowanie funkcji 

Zdefinujmy funkcje wykorzystywane podczas analizy

#### Odczytywanie Sekwencji DNA


Aby analizować sekwencje DNA, musimy najpierw załadować dane z pliku FASTA.

Funkcja *read_fasta* służy do wczytywania sekwencji DNA z pliku w formacie FASTA do słownika. 

**Parametry:**
*file_path:* Ścieżka do pliku FASTA z sekwencją DNA.


**Zwraca:**
*fadict:* Słownik, w którym kluczami są nazwy sekwencji, a wartościami są odpowiadające im sekwencje DNA.


In [2]:
def read_fasta(file_path):
    fadict = {}
    with open(file_path) as file:
        idx = ""
        for line in file:
            if line.startswith(">"):
                idx = line.strip().split()[0][1:]
                fadict[idx] = ""
            else:
                fadict[idx] += line.strip()
    return fadict

#### Analiza w ruchomych oknach

Jedną z podstawowych technik analizy sekwencji jest analiza w ruchomych oknach (sliding windows). Polega ona na podziale sekwencji na małe, nakładające się segmenty (okna) i wykonywaniu obliczeń w każdym z nich.

Funkcja *sliding_window* wykonuje analizę ruchomych okien na sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*size:* Rozmiar okna.
*step:* Krok przesuwania okna.
*func:* Funkcja, która ma zostać zastosowana do każdego okna.

**Zwraca:**
Lista wyników analizy okien, gdzie każdy wynik zawiera informacje o początku i końcu okna oraz wynik funkcji *func* na tym oknie.


In [3]:
def sliding_window(seq, size, step, func):
    results = []
    for start in range(0, len(seq) - size + 1, step):
        subseq = seq[start:start + size]
        results.append([start, start + size, func(subseq)])
    return results

#### Obliczanie Zawartości GC

**Wprowadzenie:**
Zawartość GC to miara proporcji zasad guaniny (G) i cytozyny (C) w sekwencji DNA. 

Funkcja *gc_content* Oblicza zawartość GC w danej sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.

**Zwraca:**
Procentowy udział GC w sekwencji.


In [4]:

def gc_content(seq):
    gc_count = sum(1 for base in seq if base.lower() in ["g", "c"])
    return gc_count / len(seq)

#### Obliczanie Skosu GC

Asymetria GC (GC skew) określa różnicę w zawartości guaniny i cytozyny na jednej nici DNA. 

Funkcja *gc_skew* oblicza asymetrię GC w danej sekwencji DNA.
**Parametry:**

*seq:* Analizowana sekwencja DNA.

**Zwraca:**
Asymetrię GC w sekwencji.


In [5]:
def gc_skew(seq):
    g = seq.lower().count("g")
    c = seq.lower().count("c")
    return (g - c) / (g + c) if (g + c) > 0 else 0 

#### Skalowanie Min-Max

W niektórych przypadkach ważne jest przeskalowanie naszych danych tak, aby mieściły się w określonym zakresie, na przykład [0, 1].

Funkcja *rescale_range* wykonuje przeskalowanie wartości x z zakresu [min_x, max_x] na inny zakres [i, j].

**Parametry:**
*x:* Wartość do przeskalowania.
*min_x:* Dolny zakres oryginalnych wartości.
*max_x:* Górny zakres oryginalnych wartości.
*i:* Dolny zakres przeskalowanych wartości.
*j:* Górny zakres przeskalowanych wartości.

**Zwraca:**
Przeskalowaną wartość *x* w nowym zakresie [i, j].


In [6]:
def rescale_range(x, min_x, max_x, i=0, j=1):
    return(i+((x-min_x)*(j-i)/(max_x-min_x)))

#### Analiza Częstości K-merów

K-mery to sekwencje o długości k w sekwencji DNA. Analiza częstości k-merów może dostarczyć wglądu w charakterystykę sekwencji. 

Funkcja *kmer_freq* analizuje sekwencję DNA w poszukiwaniu wystąpień k-merów i zlicza ich częstość.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*k:* Długość k-mera (domyślnie 20).

**Zwraca:**
Słownik, w którym kluczami są k-mery, a wartościami liczba ich wystąpień w sekwencji.

In [7]:
def kmer_freq(seq, k=20):
    kmer_dic = {}
    for i in range(len(seq) - k + 1):
        subseq = seq[i:i + k]
        kmer_dic[subseq] = kmer_dic.get(subseq, 0) + 1
    return kmer_dic

#### Pobieranie Pozycji K-merów

Funkcja *kmer_pos* znajduje pozycje wystąpień danego k-mera w sekwencji DNA.

**Parametry:**
*seq:* Analizowana sekwencja DNA.
*kmer:* K-mer, którego pozycje chcemy znaleźć.

**Zwraca:**
Lista pozycji wystąpień k-mera, gdzie każda pozycja to krotka (start, stop).


In [8]:
def kmer_pos(seq, kmer):
    pos = []
    k = len(kmer)
    start = 0
    while True:
        start = seq.find(kmer, start)
        if start == -1:
            break
        pos.append((start, start + k))
        start += 1
    return pos


#### Scalanie Nachodzących Zakresów 

Funkcja merge_overlaps łączy nakładające się zakresy (pary start-stop) w sekwencjach. Funkcja wymaga posortowanych zakresów wejściowych. 

**Parametry:**
*ranges:* Lista zakresów do połączenia.

**Zwraca:**
Złączone zakresy, które eliminują nakładające się fragmenty i tworzą spójne obszary.


In [9]:
def merge_overlaps(ranges):
    if not ranges:
        return []
    
    ranges = sorted(ranges)
    merged = [ranges[0]]
    
    for current in ranges[1:]:
        last = merged[-1]
        if current[0] <= last[1]:  # Overlapping intervals
            merged[-1] = [last[0], max(last[1], current[1])]
        else:
            merged.append(current)
    return merged


## Przygotowanie genomu Wejściowego

W tej sekcji przygotujemy dane genomu do analizy. Pobierzemy i rozpakujemy plik FASTA z sekwencją genomu i zmienimy jego nazwę na bardziej intuicyjną. Proszę zmienić genom na wybrany przez siebie kompletny genom bakteryjny, opublikowany nie wcześniej niż w 2018 roku. Genom można pobrać z bazy danych [NCBI Genomes](https://www.ncbi.nlm.nih.gov/datasets/genomes/). 


In [10]:
!wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/027/325/GCF_000027325.1_ASM2732v1/GCF_000027325.1_ASM2732v1_genomic.fna.gz

--2025-10-28 13:14:08--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/027/325/GCF_000027325.1_ASM2732v1/GCF_000027325.1_ASM2732v1_genomic.fna.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.11, 130.14.250.12, 130.14.250.7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.11|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 165775 (162K) [application/x-gzip]
Saving to: ‘GCF_000027325.1_ASM2732v1_genomic.fna.gz’

GCF_000027325.1_ASM 100%[===================>] 161.89K   522KB/s    in 0.3s    

2025-10-28 13:14:09 (522 KB/s) - ‘GCF_000027325.1_ASM2732v1_genomic.fna.gz’ saved [165775/165775]



In [11]:
!gunzip GCF_000027325.1_ASM2732v1_genomic.fna.gz 

In [12]:
!mv GCF_000027325.1_ASM2732v1_genomic.fna mycoplasma.fa

In [10]:
in_fasta = "GENOME_thin.fasta"
fadict = read_fasta(in_fasta)

Analizę przeprowadzimy jedynie dla chromosomu bakteryjnego, więc jeżeli w złożeniu znajdują się jakiekolwiek dodatkowe sekwencje (np. plazmidy) proszę je zignorować, i do dalszych kroków wykorzystać jedynie sekwencję największej cząsteczki.

In [11]:
fadict.keys()

dict_keys(['ptg000007l', 'ptg000003l', 'ptg000012l', 'ptg000008l', 'ptg000002l', 'ptg000016l', 'ptg000018l', 'ptg000014l', 'ptg000011l', 'ptg000015l', 'ptg000010l', 'ptg000004l', 'ptg000001l', 'ptg000009l', 'ptg000013l', 'ptg000017l'])

In [12]:
for key in fadict.keys():
    print(key, len(fadict[key]))

ptg000007l 4962671
ptg000003l 1805757
ptg000012l 2044213
ptg000008l 2324804
ptg000002l 2259285
ptg000016l 2349775
ptg000018l 2524214
ptg000014l 2808376
ptg000011l 3109985
ptg000015l 3330951
ptg000010l 4100071
ptg000004l 3736295
ptg000001l 3711849
ptg000009l 4221094
ptg000013l 4262825
ptg000017l 4416379


In [16]:
chromnames = list(fadict.keys())
chromnames

['ptg000007l',
 'ptg000003l',
 'ptg000012l',
 'ptg000008l',
 'ptg000002l',
 'ptg000016l',
 'ptg000018l',
 'ptg000014l',
 'ptg000011l',
 'ptg000015l',
 'ptg000010l',
 'ptg000004l',
 'ptg000001l',
 'ptg000009l',
 'ptg000013l',
 'ptg000017l']

In [18]:
for chromname in chromnames:
    seq = fadict[chromname]
    seq_length = len(seq)
    print(seq_length)

4962671
1805757
2044213
2324804
2259285
2349775
2524214
2808376
3109985
3330951
4100071
3736295
3711849
4221094
4262825
4416379


### Analiza Zawartości GC z Wykorzystaniem Okien


W tej sekcji wykorzystamy funkcję *sliding_window*, aby obliczyć zawartość GC w sekwencji genomu korzystając z okien o zdefiniowanej wielkości i kroku. Poeksperymentuj z wielkością okna oraz kroku, aby zobaczyć jak parametry te wpływają na wynik. Poniważ Circos wymaga, aby zakresy danych nie nakładały się, w ostatnim przebiegu ustaw długość kroku równą wielkości okna.


In [20]:
w_size = 50000
w_step = 50000

In [53]:
from copy import deepcopy
gcdf_dict = {}

for chromname in chromnames:
    seq = fadict[chromname]
    seq_length = len(seq)
    print(seq_length)
    
    gc = sliding_window(seq, w_size, w_step, gc_content)

    gcdf = pd.DataFrame(gc, columns=["start", "stop", "gc"])
    print(gcdf)
    gcdf_dict[chromname]= gcdf

print(len(gcdf_dict))


4962671
      start     stop       gc
0         0    50000  0.47702
1     50000   100000  0.49118
2    100000   150000  0.47806
3    150000   200000  0.46692
4    200000   250000  0.45054
..      ...      ...      ...
94  4700000  4750000  0.51006
95  4750000  4800000  0.44950
96  4800000  4850000  0.50556
97  4850000  4900000  0.48292
98  4900000  4950000  0.45106

[99 rows x 3 columns]
1805757
      start     stop       gc
0         0    50000  0.51252
1     50000   100000  0.48708
2    100000   150000  0.45720
3    150000   200000  0.47562
4    200000   250000  0.48344
5    250000   300000  0.48452
6    300000   350000  0.49106
7    350000   400000  0.49168
8    400000   450000  0.47088
9    450000   500000  0.47148
10   500000   550000  0.50730
11   550000   600000  0.49122
12   600000   650000  0.50270
13   650000   700000  0.49546
14   700000   750000  0.50182
15   750000   800000  0.49460
16   800000   850000  0.42122
17   850000   900000  0.42900
18   900000   950000  0.49362
1

#### Zawartość GC z poziomu terminala

Do szybkiego obliczenia zawartości GC możemy wykorzystać narzędzie **fx2tab** z pakietu **seqkit** korzystając z flag **-n -B GC**. Argumenty flagi **-B** określają nukleotydy, których relatywny udział w sekwencji chcemy określić. Alternatywnie możemy posłużyć się równoważnym **-n -g**

In [20]:
!seqkit fx2tab {in_fasta} -n -B GC

NC_000908.2 Mycoplasmoides genitalium G37, complete sequence	31.69


#### Seqkit umożliwia również analizę w oknie

In [21]:
! cat {in_fasta}| \
 seqkit sliding -s 5000 -W 5000 | \
 seqkit fx2tab -n -g | head

NC_000908.2_sliding:1-5000	28.28
NC_000908.2_sliding:5001-10000	30.54
NC_000908.2_sliding:10001-15000	26.64
NC_000908.2_sliding:15001-20000	29.14
NC_000908.2_sliding:20001-25000	29.52
NC_000908.2_sliding:25001-30000	29.06
NC_000908.2_sliding:30001-35000	30.88
NC_000908.2_sliding:35001-40000	30.32
NC_000908.2_sliding:40001-45000	30.24
NC_000908.2_sliding:45001-50000	31.52


### Wizualizacja Zawartości GC

Dane z poprzednich sekcji zostaną użyte do utworzenia wykresu liniowego, który pokaże zmiany zawartości GC w sekwencji genomu.


In [31]:
for chromname, gcdf in gcdf_dict.items():
    gcplot = (ggplot(gcdf, aes(x="start", y="gc")) + geom_line() + theme_minimal())
    print(gcplot)
    gcplot.save(f"gc_plot_{chromname}.png", width=8, height = 6)
    

<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000007l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000003l.png


<ggplot: (640 x 480)>


...

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000009l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000013l.png


<ggplot: (640 x 480)>


/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: gc_plot_ptg000017l.png


### Sformatowanie danych do narzędzia Circos

Dane dotyczące zawartości GC zwizualizujemy w postaci histogramu. W tym celu Circos wymaga danych w formacie (bez nagłówka):

|chromosom|start|stop|wartość|kolor|
|--|--|--|--|--|
|Chr1|	0	|100|	1.629407336396254|	fill_color=red|
|Chr1	|100|	200|	1.5523160283080437|	fill_color=red|


#### Dodanie Informacji o Chromosomie


W tej sekcji przypiszemy do naszych danych identyfikator chromosomu. 

In [54]:
gcdf_dict_insert = deepcopy(gcdf_dict)

for chromname, gcdf in gcdf_dict.items():
    gcdf.insert(0,"chr",chromname)
    gcdf_dict_insert[chromname] = gcdf
    print(gcdf)

           chr    start     stop       gc
0   ptg000007l        0    50000  0.47702
1   ptg000007l    50000   100000  0.49118
2   ptg000007l   100000   150000  0.47806
3   ptg000007l   150000   200000  0.46692
4   ptg000007l   200000   250000  0.45054
..         ...      ...      ...      ...
94  ptg000007l  4700000  4750000  0.51006
95  ptg000007l  4750000  4800000  0.44950
96  ptg000007l  4800000  4850000  0.50556
97  ptg000007l  4850000  4900000  0.48292
98  ptg000007l  4900000  4950000  0.45106

[99 rows x 4 columns]
           chr    start     stop       gc
0   ptg000003l        0    50000  0.51252
1   ptg000003l    50000   100000  0.48708
2   ptg000003l   100000   150000  0.45720
3   ptg000003l   150000   200000  0.47562
4   ptg000003l   200000   250000  0.48344
5   ptg000003l   250000   300000  0.48452
6   ptg000003l   300000   350000  0.49106
7   ptg000003l   350000   400000  0.49168
8   ptg000003l   400000   450000  0.47088
9   ptg000003l   450000   500000  0.47148
10  ptg0000

#### Przypisanie Kolorów do Zakresów GC
W tej części przypiszemy kolory do różnych zakresów zawartości GC. W zależności od wartości zawartości GC, zakresy te otrzymają różne kolory na wykresie. Kolory określimy w formacie "R,G,B".


In [55]:
gcdf_dict_final = deepcopy(gcdf_dict_insert)

for chromname, gcdf in gcdf_dict_insert.items():
    # Definiujemy przedziały (bins) oraz odpowiadające im etykiety kolorów
    bins = [0, 0.25, 0.35, float('inf')]
    colors = ["fill_color=103,201,129", "fill_color=201,193,103", "fill_color=204,116,92"]

    # Używamy pd.cut do zaklasyfikowania wartości w kolumnie 'gc' i przypisania odpowiednich etykiet na podstawie przedziałów
    gcdf['col'] = pd.cut(gcdf['gc'], bins=bins, labels=colors, right=True)
    gcdf_dict_final[chromname] = gcdf
    print(gcdf)

           chr    start     stop       gc                    col
0   ptg000007l        0    50000  0.47702  fill_color=204,116,92
1   ptg000007l    50000   100000  0.49118  fill_color=204,116,92
2   ptg000007l   100000   150000  0.47806  fill_color=204,116,92
3   ptg000007l   150000   200000  0.46692  fill_color=204,116,92
4   ptg000007l   200000   250000  0.45054  fill_color=204,116,92
..         ...      ...      ...      ...                    ...
94  ptg000007l  4700000  4750000  0.51006  fill_color=204,116,92
95  ptg000007l  4750000  4800000  0.44950  fill_color=204,116,92
96  ptg000007l  4800000  4850000  0.50556  fill_color=204,116,92
97  ptg000007l  4850000  4900000  0.48292  fill_color=204,116,92
98  ptg000007l  4900000  4950000  0.45106  fill_color=204,116,92

[99 rows x 5 columns]
           chr    start     stop       gc                    col
0   ptg000003l        0    50000  0.51252  fill_color=204,116,92
1   ptg000003l    50000   100000  0.48708  fill_color=204,116,92
2 

#### Zapisywanie Danych

Teraz, gdy przeprowadziliśmy analizę zawartości GC, przypisaliśmy kolory i dodaliśmy informacje o chromosomie, zapiszemy te dane do pliku.

In [39]:
for chromname, gcdf in gcdf_dict_final.items():
    gcdf.to_csv(f"gc_content_{chromname}.histo", sep="\t", index=False, header=False)

### Analiza asymetrii GC z Wykorzystaniem Okien

W tej sekcji przeprowadzimy analizę asymetrii GC. Tak jak w przypadku zawartości GC przeprowadźmy analizę dla różnych wartości długości okna oraz kroku.


In [58]:
skew_data = {}

for chromname in chromnames:
    seq = fadict[chromname]
    #seq_length = len(seq)
    #print(seq_length)

    skew = sliding_window(seq, w_size, w_step, gc_skew)
    skewdf = pd.DataFrame(skew, columns=["start", "stop", "skew"])
    skew_data[chromname] = skewdf
    print(skewdf)

      start     stop      skew
0         0    50000  0.010524
1     50000   100000  0.005253
2    100000   150000 -0.025394
3    150000   200000  0.024672
4    200000   250000 -0.031651
..      ...      ...       ...
94  4700000  4750000  0.011097
95  4750000  4800000  0.008320
96  4800000  4850000 -0.048026
97  4850000  4900000 -0.002899
98  4900000  4950000 -0.050947

[99 rows x 3 columns]
      start     stop      skew
0         0    50000 -0.025365
1     50000   100000 -0.026361
2    100000   150000  0.002450
3    150000   200000  0.002481
4    200000   250000 -0.008522
5    250000   300000 -0.015273
6    300000   350000  0.006802
7    350000   400000  0.011390
8    400000   450000 -0.005097
9    450000   500000  0.011793
10   500000   550000 -0.020304
11   550000   600000  0.005741
12   600000   650000  0.010304
13   650000   700000  0.006822
14   700000   750000 -0.008808
15   750000   800000  0.025556
16   800000   850000 -0.011348
17   850000   900000 -0.037669
18   900000   95

#### Asymetria GC przy pomocy seqkit

In [27]:
!cat {in_fasta}| \
seqkit sliding -s 5000 -W 5000 | \
seqkit fx2tab -n -G|head 

NC_000908.2_sliding:1-5000	10.47
NC_000908.2_sliding:5001-10000	11.33
NC_000908.2_sliding:10001-15000	1.80
NC_000908.2_sliding:15001-20000	7.89
NC_000908.2_sliding:20001-25000	11.92
NC_000908.2_sliding:25001-30000	13.15
NC_000908.2_sliding:30001-35000	-3.50
NC_000908.2_sliding:35001-40000	2.37
NC_000908.2_sliding:40001-45000	8.60
NC_000908.2_sliding:45001-50000	2.41


#### Wizualizacja asymetrii GC

Przygotowane dane asymetrii GC zostaną użyte do utworzenia wykresu liniowego.


In [59]:
for chromname, skewdf in skew_data.items():
    skewplot = (ggplot(skewdf, aes(x="start", y="skew")) + geom_line()) + theme_minimal()
    skewplot.save(f"skew_plot_{chromname}", width=8, height=6)

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000007l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000003l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_ptg000012l
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/

### Sformatowanie danych do narzędzia Circos

Dane dotyczące zawartości GC zwizualizujemy w postaci wykresu punktowego (scatterplot). Format danych dla tego typu wykresu jest tożsamy z powyższym, rodzaj wykresu określamy w pliku konfiguracyjnym, co zrobimy w dalszej części.


#### Dodawanie Informacji o Chromosomie


W tej sekcji przypiszemy do naszych danych identyfikator chromosomu. 

In [60]:
skew_data_insert = deepcopy(skew_data)

for chromname, skewdf in skew_data.items():

    skewdf.insert(0,"chr",chromname)
    skew_data_insert[chromname] = skewdf
    print(skewdf)


           chr    start     stop      skew
0   ptg000007l        0    50000  0.010524
1   ptg000007l    50000   100000  0.005253
2   ptg000007l   100000   150000 -0.025394
3   ptg000007l   150000   200000  0.024672
4   ptg000007l   200000   250000 -0.031651
..         ...      ...      ...       ...
94  ptg000007l  4700000  4750000  0.011097
95  ptg000007l  4750000  4800000  0.008320
96  ptg000007l  4800000  4850000 -0.048026
97  ptg000007l  4850000  4900000 -0.002899
98  ptg000007l  4900000  4950000 -0.050947

[99 rows x 4 columns]
           chr    start     stop      skew
0   ptg000003l        0    50000 -0.025365
1   ptg000003l    50000   100000 -0.026361
2   ptg000003l   100000   150000  0.002450
3   ptg000003l   150000   200000  0.002481
4   ptg000003l   200000   250000 -0.008522
5   ptg000003l   250000   300000 -0.015273
6   ptg000003l   300000   350000  0.006802
7   ptg000003l   350000   400000  0.011390
8   ptg000003l   400000   450000 -0.005097
9   ptg000003l   450000   50000

#### Przypisanie Kolorów do asymetrii GC
W tej części przypiszemy kolory do różnych zakresów asymetrii GC. Skorzystamy z domyślnie zdefiniowanych kolorów.

In [61]:
skew_data_final = deepcopy(skew_data_insert)

for chromname, skewdf in skew_data_insert.items():
    skewdf["col"] = "fill_color=blue"
    skewdf.loc[skewdf["skew"]>0, "col"] = "fill_color=red"
    skew_data_final[chromname] = skewdf
    print(skewdf)

           chr    start     stop      skew              col
0   ptg000007l        0    50000  0.010524   fill_color=red
1   ptg000007l    50000   100000  0.005253   fill_color=red
2   ptg000007l   100000   150000 -0.025394  fill_color=blue
3   ptg000007l   150000   200000  0.024672   fill_color=red
4   ptg000007l   200000   250000 -0.031651  fill_color=blue
..         ...      ...      ...       ...              ...
94  ptg000007l  4700000  4750000  0.011097   fill_color=red
95  ptg000007l  4750000  4800000  0.008320   fill_color=red
96  ptg000007l  4800000  4850000 -0.048026  fill_color=blue
97  ptg000007l  4850000  4900000 -0.002899  fill_color=blue
98  ptg000007l  4900000  4950000 -0.050947  fill_color=blue

[99 rows x 5 columns]
           chr    start     stop      skew              col
0   ptg000003l        0    50000 -0.025365  fill_color=blue
1   ptg000003l    50000   100000 -0.026361  fill_color=blue
2   ptg000003l   100000   150000  0.002450   fill_color=red
3   ptg000003l   

#### Zapisywanie danych

In [46]:
for chromname, skewdf in skew_data_final.items():
    skewdf.to_csv(f"gc_skew_{chromname}.histo", sep="\t", index=False, header=False)

### Obliczanie Kumulatywnej Asymetrii GC

Kumulatywna asymetria GC to kumulatywna suma skosu GC kolejnych oknach. W tej sekcji obliczymy i przeskalujemy te wartości, aby uzyskać wyniki w określonym zakresie, co ułatwi nam zwizualizowanie asymetrii GC i kumulatywnej asymetrii GC na jednym wykresie.


In [62]:
skew_data_cuml = deepcopy(skew_data_final)

for chromname, skewdf in skew_data_final.items():
    skew_cuml = []
    cumul = 0

    # Dla każdego okna dodajemy skew poprzedniego okna
    for idx in range(skewdf.shape[0]):
        cumul += skewdf.iloc[idx,]["skew"]
        skew_cuml.append(cumul)
    
    skew_cuml = np.array(skew_cuml)
    skew_data_cuml[chromname] = [skewdf, skew_cuml]

#### Przeskalowanie wartości kumulatywnej asymetrii GC do zakresu 0-1

In [64]:
skew_data_cuml_scaled = deepcopy(skew_data_cuml)

for chromname, skew_df_cuml in skew_data_cuml.items():
    skewdf = skew_df_cuml[0]
    skew_cuml = skew_df_cuml[1]

    skew_cuml_scaled = rescale_range(skew_cuml, min(skew_cuml), max(skew_cuml))
    print(skew_cuml_scaled)
    skew_data_cuml_scaled[chromname] = [skewdf, skew_cuml_scaled]


[0.2035306  0.23105314 0.09799366 0.22727015 0.06142769 0.0461883
 0.16006133 0.2511428  0.49011882 0.59955219 0.62365127 0.6479537
 0.71094061 0.92925308 1.         0.8748154  0.61519081 0.56603755
 0.47855801 0.17290482 0.09097155 0.20070896 0.30943473 0.46078766
 0.32052787 0.38919357 0.45165578 0.38336614 0.44844813 0.42633584
 0.44865611 0.39520239 0.26701959 0.32341641 0.29551002 0.3369777
 0.27745688 0.33547334 0.29955163 0.38753076 0.37618621 0.39174054
 0.3652626  0.35056517 0.41250658 0.43179654 0.23039243 0.22997092
 0.17109272 0.23002659 0.16219014 0.22280006 0.25077955 0.24293848
 0.15283247 0.14321527 0.04686691 0.14982198 0.05302698 0.04930616
 0.06094767 0.05217304 0.05281836 0.11160281 0.14457855 0.15245721
 0.         0.07435391 0.17456918 0.18061704 0.23871749 0.25287609
 0.31905154 0.295224   0.35214606 0.37241073 0.3436521  0.41032913
 0.38160963 0.34603192 0.32997804 0.38682213 0.35540858 0.41053211
 0.44652307 0.37930304 0.40402403 0.29628165 0.40631977 0.3988010

#### Wizualizacja asymetrii GC i kumulatywnej asymetrii GC

W celu wizualizacji dołączymy kolumnę *skew_cuml* do ramki *skewdf*


In [66]:
skew_data_final_scaled = {}

for chromname, skew_df_cuml in skew_data_cuml_scaled.items():
    skewdf = skew_df_cuml[0]
    skew_cuml_scaled = skew_df_cuml[1]

    skewdf["skew_cuml"] = skew_cuml_scaled
    print(skewdf)
    skew_data_final_scaled[chromname] = skewdf

           chr    start     stop      skew              col  skew_cuml
0   ptg000007l        0    50000  0.010524   fill_color=red   0.203531
1   ptg000007l    50000   100000  0.005253   fill_color=red   0.231053
2   ptg000007l   100000   150000 -0.025394  fill_color=blue   0.097994
3   ptg000007l   150000   200000  0.024672   fill_color=red   0.227270
4   ptg000007l   200000   250000 -0.031651  fill_color=blue   0.061428
..         ...      ...      ...       ...              ...        ...
94  ptg000007l  4700000  4750000  0.011097   fill_color=red   0.618276
95  ptg000007l  4750000  4800000  0.008320   fill_color=red   0.661872
96  ptg000007l  4800000  4850000 -0.048026  fill_color=blue   0.410229
97  ptg000007l  4850000  4900000 -0.002899  fill_color=blue   0.395039
98  ptg000007l  4900000  4950000 -0.050947  fill_color=blue   0.128092

[99 rows x 6 columns]
           chr    start     stop      skew              col  skew_cuml
0   ptg000003l        0    50000 -0.025365  fill_color

#### Zmiana formatu z szerokiego (wide) na długi (long)

In [67]:
skew_data_final_molten = {}

for chromname, skewdf in skew_data_final_scaled.items():
    skewdf_molten = pd.melt(skewdf, id_vars=["start","stop"], value_vars=["skew","skew_cuml"])
    
    skew_data_final_molten[chromname] = skewdf_molten
    print(skewdf_molten)

       start     stop   variable     value
0          0    50000       skew  0.010524
1      50000   100000       skew  0.005253
2     100000   150000       skew -0.025394
3     150000   200000       skew  0.024672
4     200000   250000       skew -0.031651
..       ...      ...        ...       ...
193  4700000  4750000  skew_cuml  0.618276
194  4750000  4800000  skew_cuml  0.661872
195  4800000  4850000  skew_cuml  0.410229
196  4850000  4900000  skew_cuml  0.395039
197  4900000  4950000  skew_cuml  0.128092

[198 rows x 4 columns]
      start     stop   variable     value
0         0    50000       skew -0.025365
1     50000   100000       skew -0.026361
2    100000   150000       skew  0.002450
3    150000   200000       skew  0.002481
4    200000   250000       skew -0.008522
..      ...      ...        ...       ...
67  1550000  1600000  skew_cuml  0.170726
68  1600000  1650000  skew_cuml  0.499781
69  1650000  1700000  skew_cuml  0.713511
70  1700000  1750000  skew_cuml  0.95781

#### Wizualizacja

Dla wielu bakteryjnych genomów bakteryjnych wykres asymetrii GC przyjmuje charakterystyczny kształt, z punktami przełamania odpowiadającym miejscom inicjacji i terminacji replikacji (ori). Jest to bardzo interesujący przykład analizy prostych parametrów sekwencji mającej bezpośrednie przełożenie na biologię. 

Dla zainteresowanych tematem:
- [Analyzing genomes with cumulative skew diagrams](https://academic.oup.com/nar/article/26/10/2286/1030593)
- [Asymmetric substitution patterns: a review of possible underlying mutational or selective mechanisms](https://www.sciencedirect.com/science/article/pii/S0378111999002978?via%3Dihub)
- [SkewIT: The Skew Index Test for large-scale GC Skew analysis of bacterial genomes](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7717575)

In [74]:
for chromname, skewdf_molten in skew_data_final_molten.items():
    skewplot = (ggplot(skewdf_molten, aes(x="start", y="value", color="variable")) + geom_line() +
           theme_minimal() + ylab("") + xlab("Position [bp]") + scale_colour_discrete(labels=["GC skew", "Cumulative GC skew" ]) +
           labs(colour=""))
    skewplot.save(f"skew_plot_molten_{chromname}.png", width=8, height=6)

/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000007l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000003l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: skew_plot_molten_ptg000012l.png
/opt/miniconda3/envs/jupyternotebook/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 8 x 6 in image.
/opt/miniconda3/envs/jupyternoteb

### Sformatowanie danych do narzędzia Circos

Dane dotyczące kumulatywnej zawartości GC zwizualizujemy w postaci wykresu liniowego. Format danych dla tego typu wykresu, tak jak w poprzednim przypadku nie ulega zmianie.

In [37]:
for chromname, skewdf_molten in skew_data_final_molten.items():
    print(skewdf_molten)

,chr,start,stop,skew,col,skew_cuml
0,NC_000908.2,0,5000,0.104668,fill_color=red,0.000000
1,NC_000908.2,5000,10000,0.113294,fill_color=red,0.043697
2,NC_000908.2,10000,15000,0.018018,fill_color=red,0.050646
3,NC_000908.2,15000,20000,0.078929,fill_color=red,0.081089
4,NC_000908.2,20000,25000,0.119241,fill_color=red,0.127080
...,...,...,...,...,...,...
111,NC_000908.2,555000,560000,0.026263,fill_color=red,0.148306
112,NC_000908.2,560000,565000,-0.059786,fill_color=blue,0.125247
113,NC_000908.2,565000,570000,-0.045847,fill_color=blue,0.107564
114,NC_000908.2,570000,575000,-0.022279,fill_color=blue,0.098971


In [76]:
c_skew_data_final_molten = {}

for chromname, skewdf_molten in skew_data_final_molten.items():
    print(skewdf_molten.columns)
    c_skewdf = skewdf_molten[["chr","start","stop","skew_cuml"]].copy()
    c_skewdf['col'] = "color=pink"
    c_skew_data_final_molten[chromname] = c_skewdf

    print(c_skewdf)

Index(['start', 'stop', 'variable', 'value'], dtype='object')


KeyError: "['chr', 'skew_cuml'] not in index"

In [71]:
for chromname, c_skewdf in c_skew_data_final_molten.items():
    print(chromname)
    c_skewdf.to_csv(f"gc_skew_cumul_{chromname}.histo", sep="\t", index=False, header=False)

ptg000007l
ptg000003l
ptg000012l
ptg000008l
ptg000002l
ptg000016l
ptg000018l
ptg000014l
ptg000011l
ptg000015l
ptg000010l
ptg000004l
ptg000001l
ptg000009l
ptg000013l
ptg000017l


## Wizualizacja połączeń oraz kafelków na wykresie typu Circos

Aby zapoznać się z elementami typu "links" (połączenia) oraz "tile" (kafelek) na wykresie Circos wykorzystamy pozycje najliczniej wystepującego 20-meru. W tym celu najpierw zliczymy wystąpienia obecnych w sekwencji 20-merów.

In [40]:
kmers = kmer_freq(seq)
kmers = sorted(kmers.items(), key=lambda n: n[1], reverse=True)
kmers[0]

('TAGTAGTAGTAGTAGTAGTA', 29)

### Znajdowanie Pozycji 20-mera o Największej Częstości

W tej sekcji zidentyfikujemy 20-mer o największej częstości w sekwencji genomu i znajdziemy jego pozycje na genomie.


In [41]:
max_kmer = kmers[0][0]
kpos = kmer_pos(seq, max_kmer)
kpos

[(169475, 169495),
 (169478, 169498),
 (169481, 169501),
 (169484, 169504),
 (169487, 169507),
 (169490, 169510),
 (169493, 169513),
 (169496, 169516),
 (169499, 169519),
 (169502, 169522),
 (224532, 224552),
 (224535, 224555),
 (227128, 227148),
 (227131, 227151),
 (227134, 227154),
 (227137, 227157),
 (227140, 227160),
 (227143, 227163),
 (227146, 227166),
 (351452, 351472),
 (351455, 351475),
 (351458, 351478),
 (351461, 351481),
 (351464, 351484),
 (429304, 429324),
 (429307, 429327),
 (429310, 429330),
 (429313, 429333),
 (429316, 429336)]

#### Połączenie nakładających się zakresów

In [42]:
kpos_merged = merge_overlaps(kpos)
kpos_merged

[[169475, 169522],
 [224532, 224555],
 [227128, 227166],
 [351452, 351484],
 [429304, 429336]]

#### Przygotowanie Danych do Circos

Teraz, gdy mamy pozycje 20-merów, które chcemy wyświetlić w Circos, przygotujemy dane, które będą zgodne z tym narzędziem do wizualizacji.
Pozycje 20-merów, które uwidocznimy w postaci kafelków sformatujemy w sposób tożsamy z poprzednimi danymi.


In [43]:
with open("max_kmer.histo", "w") as out:
    col = "darkblue"
    for line in kpos_merged:
        lineout = [chromname] + line + [f"color={col}"]
        print(lineout)
        out.write("\t".join([str(x) for x in lineout]) + "\n")

['NC_000908.2', 169475, 169522, 'color=darkblue']
['NC_000908.2', 224532, 224555, 'color=darkblue']
['NC_000908.2', 227128, 227166, 'color=darkblue']
['NC_000908.2', 351452, 351484, 'color=darkblue']
['NC_000908.2', 429304, 429336, 'color=darkblue']


### Wizualizacja połączeń (links)

Połączenia w obrębie wykresów Circos wykorzystywane są m.in. do wizualizacji regionów homologicznych. Tutaj jako przykład wykorzystamy znane nam pozycje najliczniejszego 20-meru.

Połączenia definiujemy określając łączone pary zakresów w pojedynczych liniach:

|chrom1|start1|stop1|chrom2|start2|stop2|parametry
|--|--|--|--|--|--|--|
|	0|	169475|	169522	|	0|	224535	|224555	|color=blue
|	0|	33243	|35664	|	1|	442	|816	|color=blue


In [44]:
# Create an empty DataFrame to store the results
kmer_links_df = pd.DataFrame(columns=["Chromname1", "Start1", "Stop1","Chromname2", "Start2", "Stop2","Color"])
col = "blue"

for i in range(len(kpos_merged)):
    for j in range(i,len(kpos_merged)):
        if i != j:
            lineout = [chromname, *kpos_merged[i], chromname, *kpos_merged[j], f"color={col}"]
            kmer_links_df.loc[len(kmer_links_df)] = lineout

kmer_links_df

,Chromname1,Start1,Stop1,Chromname2,Start2,Stop2,Color
0,NC_000908.2,169475,169522,NC_000908.2,224532,224555,color=blue
1,NC_000908.2,169475,169522,NC_000908.2,227128,227166,color=blue
2,NC_000908.2,169475,169522,NC_000908.2,351452,351484,color=blue
3,NC_000908.2,169475,169522,NC_000908.2,429304,429336,color=blue
4,NC_000908.2,224532,224555,NC_000908.2,227128,227166,color=blue
5,NC_000908.2,224532,224555,NC_000908.2,351452,351484,color=blue
6,NC_000908.2,224532,224555,NC_000908.2,429304,429336,color=blue
7,NC_000908.2,227128,227166,NC_000908.2,351452,351484,color=blue
8,NC_000908.2,227128,227166,NC_000908.2,429304,429336,color=blue
9,NC_000908.2,351452,351484,NC_000908.2,429304,429336,color=blue


In [45]:
# Write the DataFrame to a file
kmer_links_df.to_csv("max_kmer.links", sep="\t", index=False, header=False)

### Pliki konfiguracyjne Circos

Aby stworzyć wykres, oprócz danych, musimy przygotować kilka plików konfiguracyjnych, w których definiujemy sposób w jaki wyświetlone mają być nasze dane.

Są to:

#### Kariotyp

Określamy w nim podstawowe informacje dotyczące segmentów (chromosomów), które chcemy zwizualizować. Format:

|chr|- |ID (w danych)|etykieta|początek|koniec|kolor 
|--|--|--|--|--|--|--|
|chr|	-|	1|	chromosome1|	0|	580076|	purple|

W tym przykładzie kariotyp jest mało skomplikowany, i możemy go w prosty sposób wygenerować korzystając z dostepnych danych.


In [46]:
karyo = ["chr","-",chromname,"chromosome","0",str(seq_length),"purple"]
karyo

['chr', '-', 'NC_000908.2', 'chromosome', '0', '580076', 'purple']

In [47]:
with open("karyo.conf", "w") as out: out.write("\t".join(karyo)+"\n")


#### Główny plik konfiguracyjny

Definiujemy w nim elementy grafiki, oraz ich parametry. W głównym pliku konfiguracyjnym określamy lokalizację pliku z kariotypem:

>  karyotype = karyo.conf

Elementami, które możemy uwzględnić na wykresie są m.in.:

Ideogramy, czyli segmenty "chromosomów":

> \<ideogram\>
> 
> \<spacing\> default = 0.005r \</spacing\>
> 
> radius           = 0.90r 
> thickness        = 20p 
> fill             = yes
> 
> #stroke_thickness = 1
> #stroke_color     = black
> 
> 
> \</ideogram\>

Oznaczenia osi (ticks):

> \<tick> spacing        = 10000 u
>  color          = grey 
>  size           = 10p
>   \</tick>

 Wykresy prezentujące dane liczbowe:

> \<plot>
> 
> type = histogram
>  file = gc_content.histo
>   thickness = 0p
> 
> \</plot>

Linie łączące elementy wykresu:

> \<links> 
> \<link> radius = 0.8r 
> bezier_radius = 0r 
> bezier_radius_purity = 0.9
>  color = black 
>  thickness = 2 
>  file = max_kmer.links 
>  \</link> 
>  \</links>

Plik konfiguracyjny do tego ćwiczenia możemy pobrać z chmury:

In [48]:
!wget "https://drive.google.com/uc?export=download&id=1eWXYScnQnOAyd50cI3ycwpigDJFxlvSv" -O circos_conf.conf
!ls | grep conf

--2025-10-28 13:15:21--  https://drive.google.com/uc?export=download&id=1eWXYScnQnOAyd50cI3ycwpigDJFxlvSv
Resolving drive.google.com (drive.google.com)... 216.58.208.206, 2a00:1450:401b:800::200e
Connecting to drive.google.com (drive.google.com)|216.58.208.206|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1eWXYScnQnOAyd50cI3ycwpigDJFxlvSv&export=download [following]
--2025-10-28 13:15:21--  https://drive.usercontent.google.com/download?id=1eWXYScnQnOAyd50cI3ycwpigDJFxlvSv&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.38.161, 2a00:1450:401b:802::2001
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.38.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1820 (1.8K) [application/octet-stream]
Saving to: ‘circos_conf.conf’

circos_conf.conf    100%[===================>]   1.78K  --.-KB/s    in 0s 

In [49]:
!cat circos_conf.conf


########## Kariotyp - tutaj definiujemy chromosomy
karyotype = karyo.conf



<ideogram>

<spacing>
# spacing between ideograms
default = 0.005r
</spacing>

# ideogram position, thickness and fill
radius           = 0.90r
thickness        = 20p
fill             = yes

#stroke_thickness = 1
#stroke_color     = black


</ideogram>


####### Ta część w zasadzie jest skopiowana z pliku przykładowego:

<image>
<<include etc/image.conf>> # included from Circos distribution
radius* = 1500 
</image>


# RGB/HSV color definitions, color lists, location of fonts,
# fill patterns
<<include etc/colors_fonts_patterns.conf>> # included from Circos distribution

# debugging, I/O an dother system parameters
<<include etc/housekeeping.conf>> # included from Circos distribution

#<ticks> blocks to define ticks, tick labels and grids
#
# requires that chromosomes_units be defined
#
 

### Znaczniki na osiach:
<<include ticks.conf>>



<plots>


#### GC-content
<plot>

type = histogram
file = gc_content.h

#### Dodatkowe pliki konfiguracyjne 

Możemy definiować w nich te same typy elementów, co w głównym pliku konfiguracyjnym. Uwzględniamy je w głównym pliku linijką \<\<include name.conf\>\>.



In [50]:
!wget "https://drive.google.com/uc?export=download&id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_" -O ticks.conf
!ls | grep conf

--2025-10-28 13:15:25--  https://drive.google.com/uc?export=download&id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_
Resolving drive.google.com (drive.google.com)... 216.58.208.206, 2a00:1450:401b:800::200e
Connecting to drive.google.com (drive.google.com)|216.58.208.206|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_&export=download [following]
--2025-10-28 13:15:26--  https://drive.usercontent.google.com/download?id=11ZgDy-rKfUbjYmsZOSfnFJLAwBcm-C5_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.251.38.161, 2a00:1450:401b:802::2001
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.251.38.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 499 [application/octet-stream]
Saving to: ‘ticks.conf’

ticks.conf          100%[===================>]     499  --.-KB/s    in 0s      

2025-10

In [51]:
!cat ticks.conf


show_ticks          = yes
show_tick_labels    = yes

<ticks>
skip_first_label = no
skip_last_label = no
radius           = dims(ideogram,radius_outer)
multiplier       = 1 
color            = black
thickness        = 2p
size             = 20p

<tick>
skip_first_label = no
spacing        = 100000u
show_label     = yes
label_size     = 20p
label_offset   = 10p
format         = %d
suffix = " bp"
</tick>

<tick>
spacing        = 10000 u
color          = grey
size           = 10p
</tick>

</ticks>


### Wywołanie Circos


In [52]:
!circos -conf circos_conf.conf

/bin/bash: line 1: circos: command not found
